In [ ]:
!pip install tensorflow opencv-python matplotlib numpy pandas scikit-learn

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
print("TensorFlow Version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
dataset_path = "/content/drive/MyDrive/Document-Classification/dataset"

In [ ]:
import os

print(os.listdir('/content/drive/MyDrive'))

In [ ]:
import os

dataset_path = '/content/drive/MyDrive/Document_Image_Classification'

print(os.listdir(dataset_path))

In [ ]:
import os

print(os.listdir('/content/drive/MyDrive/Document_Image_Classification'))

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets list -s "RVL-CDIP"

In [ ]:
!kaggle datasets download -d uditamin/rvl-cdip-small

In [ ]:
import os
print(os.listdir())

In [ ]:
import zipfile
import os

zip_path = "/content/rvl-cdip-small.zip"
extract_path = "/content/RVL-CDIP"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Done!")
print(os.listdir(extract_path))

In [ ]:
import os

data_path = "/content/RVL-CDIP/data"

print(os.listdir(data_path))

In [ ]:
import os

for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)
    print(folder, ":", len(os.listdir(folder_path)), "images")

In [ ]:
dataset_path = "/content/RVL-CDIP/data"

import os

print(os.listdir(dataset_path))

In [ ]:
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)
    print(folder, ":", len(os.listdir(folder_path)), "images")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (224,224)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(16, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model Compiled Successfully!")

In [ ]:
history = model.fit(
  train_data,
    validation_data=val_data,
    epochs=1
)

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
model.save("document_classifier.keras")
print("Model Saved!")

In [ ]:
class_names = list(train_data.class_indices.keys())
print(class_names)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install pdf2image
!apt-get install poppler-utils -y

In [ ]:
!pdfinfo -v

In [ ]:
!apt-get update -y
!apt-get install -y poppler-utils

In [ ]:
!which pdfinfo

In [ ]:
from pdf2image import convert_from_path

pages = convert_from_path("FEE RECEIPT.pdf")

image = pages[0]
image.save("fee_receipt_page.jpg")

print("PDF converted to image")

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img(
    "fee_receipt_page.jpg",
    target_size=(224,224)
)

img_array = image.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

class_name = class_names[np.argmax(prediction)]

print("Predicted Document Type:", class_name)

In [ ]:
import numpy as np

confidence = np.max(prediction) * 100

print("Predicted Class:", class_name)
print("Confidence:", round(confidence, 2), "%")

In [ ]:
import os
print(os.listdir())

In [ ]:
import shutil

shutil.copy(
    "document_classifier.keras",
    "/content/drive/MyDrive/document_classifier.keras"
)

print("Model copied to Drive")

In [ ]:
import json

with open("class_names.json","w") as f:
    json.dump(class_names,f)

print("Class names saved")